# Older Gaussian and Student-t alternatives

These retain the earlier prior construction and common national factor. The final Gaussian remains our **reference**, not a universally superior model. The older models use the same polls/cutoffs and earlier-cycle calibration, but differ in prior construction and state relationships. This pair isolates tails within the older architecture; notebook06 isolates tails within the final architecture.

Run notebook04 (or `python run.py live`) to refresh live data first. This notebook uses the latest saved live forecast and recomputes/caches the historical comparison. See [alternative specifications](../docs/OLDER_ALTERNATIVES.md).

In [2]:
from pathlib import Path
import sys, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
ROOT=next(p for p in [Path.cwd(),Path.cwd().parent] if (p/'election_lab.py').exists())
sys.path.insert(0,str(ROOT))
import election_lab as lab
import model_portfolio as portfolio
pd.set_option('display.max_rows',40)
RUN=lab.run_logged(portfolio.run)  # Content-verified cache; no downloads.
p=pd.read_parquet(RUN/'predictions.parquet')
s=pd.read_parquet(RUN/'seats.parquet')
summary=pd.read_parquet(RUN/'summary.parquet')
cycles=pd.read_parquet(RUN/'cycle_scores.parquet')
print('Immutable comparison:',RUN.relative_to(ROOT))

MODELS=['Bayesian','Older Gaussian','Student-t research helper']

Run log: cache/logs/run_20260921T044535.468424Z.txt
Immutable comparison: cache/runs/portfolio/20260921T044535.641483Z


## Historical validation

Each held-out cycle is predicted using earlier cycles only. `matched_live` retains the research September17 cutoff; `oct31` is October31. Repeated architecture experiments reused these years: this is chronological CV, **not an untouched final test set**.

Accuracy counts races won correctly; MAE is mean absolute D−R margin error in percentage points. Brier is squared probability error (lower is better). Coverage70 is the observed percentage inside the nominal70% interval; width70 is its width in pp. Summary MAE/Brier average cycle means; accuracy/coverage pool contests.

In [3]:
display(summary[summary.first_cycle.eq(2016)&summary.model.isin(MODELS)][['scenario','model','correct','n','accuracy_pct','absolute_error_pp','brier','coverage70','width70_pp']].round(3))
display(cycles[cycles.model.isin(MODELS)][['scenario','cycle','model','n','correct','absolute_error_pp','brier']].round(3))

,scenario,model,correct,n,accuracy_pct,absolute_error_pp,brier,coverage70,width70_pp
24,matched_live,Bayesian,128,140,91.429,6.307,0.055,78.571,19.876
34,matched_live,Older Gaussian,129,140,92.143,6.808,0.059,79.286,20.897
35,matched_live,Student-t research helper,129,140,92.143,6.801,0.058,66.429,17.751
36,oct31,Bayesian,132,140,94.286,5.206,0.050,73.571,15.800
46,oct31,Older Gaussian,133,140,95.000,5.344,0.050,72.143,15.544
47,oct31,Student-t research helper,131,140,93.571,5.357,0.049,62.143,13.089


,scenario,cycle,model,n,correct,absolute_error_pp,brier
0,matched_live,2012,Bayesian,31,29.0,7.486,0.062
10,matched_live,2012,Older Gaussian,31,28.0,7.647,0.057
11,matched_live,2012,Student-t research helper,31,28.0,7.680,0.052
12,matched_live,2014,Bayesian,27,23.0,10.222,0.103
22,matched_live,2014,Older Gaussian,27,24.0,10.543,0.107
...,...,...,...,...,...,...,...
154,oct31,2022,Older Gaussian,26,26.0,5.901,0.028
155,oct31,2022,Student-t research helper,26,26.0,5.831,0.023
156,oct31,2024,Bayesian,28,27.0,4.287,0.041
166,oct31,2024,Older Gaussian,28,27.0,4.472,0.040


## Current states and final chamber totals

All four Bayesian models use joint uncertainty. Point seats count positive mean margins; expected seats sum probabilities. D control requires51. Historical chamber accounting fixes some unmodeled contests to results; see model documentation.

In [4]:
current=s[s.evidence.eq('live') & s.model.isin(MODELS)].copy()
current['D control %']=100*current.p_D_control
display(current[['model','point_D','point_R','expected_D','D control %','D_lo70','D_hi70']].round(3))
live=p[p.evidence.eq('live') & p.model.isin(MODELS)].copy()
print('D−R margin, percentage points')
display(live.pivot(index=['geography','special'],columns='model',values='margin_pp').reindex(columns=MODELS).round(2))
print('D win probability, percent')
display((100*live.pivot(index=['geography','special'],columns='model',values='p_dem')).reindex(columns=MODELS).round(1))


,model,point_D,point_R,expected_D,D control %,D_lo70,D_hi70
180,Bayesian,51,49,50.320,45.520,49.0,52.0
182,Older Gaussian,50,50,51.090,61.997,49.0,53.0
183,Student-t research helper,50,50,51.115,63.006,49.0,53.0


D−R margin, percentage points


,model,Bayesian,Older Gaussian,Student-t research helper
geography,special,,,
AK,False,-3.10,-0.77,-0.86
AL,False,-22.02,-18.24,-18.62
AR,False,-20.80,-12.03,-11.70
CO,False,17.44,14.13,14.40
DE,False,24.95,25.61,25.88
FL,True,-5.48,-4.16,-4.14
GA,False,5.22,5.94,5.90
IA,False,-3.27,-2.80,-2.80
ID,False,-25.97,-27.04,-26.77


D win probability, percent


,model,Bayesian,Older Gaussian,Student-t research helper
geography,special,,,
AK,False,30.0,45.9,44.3
AL,False,0.0,3.3,1.9
AR,False,0.2,14.4,9.5
CO,False,92.7,85.5,88.8
DE,False,99.9,99.3,98.9
FL,True,20.3,31.8,26.7
GA,False,80.8,78.3,83.6
IA,False,29.2,34.0,29.6
ID,False,0.0,0.1,0.5


## Diagnostics and provenance

The heavy-tail alternative is actual MCMC, not merely wider Gaussian intervals. Training always precedes the forecast cycle. Unknown2026 outcomes stay missing.

In [5]:
lab.verify_run(RUN)
d=pd.read_parquet(RUN/'diagnostics.parquet')
display(d[d.model.eq('Student-t research helper')].groupby('evidence')[['rhat','bulk_ess','tail_ess']].agg(['min','max']).round(3))
assert p[p.cycle.eq(2026)].actual.isna().all()

rhat         bulk_ess            tail_ess         
          min    max       min      max        min      max
evidence                                                   
frozen    1.0  1.004  3212.080  32000.0   4775.294  32000.0
live      1.0  1.002  5613.081  32000.0  10917.527  32000.0

In [6]:
# Stable, Git-friendly published report (replaced on each rerun).
from IPython.display import FileLink
import os
PUBLISHED_REPORT = ROOT/'outputs/reports/experiments/portfolio.md'
print('Published report:', PUBLISHED_REPORT.relative_to(ROOT))
display(FileLink(os.path.relpath(PUBLISHED_REPORT, Path.cwd())))


Published report: outputs/reports/experiments/portfolio.md


/Users/amir/projects/small_models/us_election_2026/outputs/reports/experiments/portfolio.md